# Q4 — ME504 Deep Learning Project
## Predicting Max Von Mises Stress

| | |
|---|---|
| **Problem** | 1a — Cantilever beam, fixed point load, variable thickness |
| **Approach 1** | Direct MLP — 226 thickness inputs → max stress |
| **Approach 2** | PCA dimensionality reduction → reduced inputs → max stress |
| **Goal** | Compare data requirements of both approaches |

---

### 📁 Expected Folder Structure
```
your_project_folder/
    Q4_Solution.ipynb    ← this file
    output.csv           (or output.xlsx)
    stress/
        stress_0.txt
        stress_1.txt
        ...
        stress_4999.txt
```

## Cell 2 — Imports & Configuration
All settings are in one place — **change paths or hyperparameters here**.

In [ ]:
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm        # notebook-style progress bars
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import r2_score

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

warnings.filterwarnings('ignore')
%matplotlib inline
plt.rcParams['figure.dpi'] = 110

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ── Paths (change if your files are elsewhere) ────────────────────────────────
BASE_DIR   = os.getcwd()                          # same folder as this notebook server
ALT_DATA_DIR = os.path.join(BASE_DIR, 'DL_data', 'prob1_data')
STRESS_DIR = os.path.join(BASE_DIR, 'stress')

# ── Dataset ───────────────────────────────────────────────────────────────────
N_SAMPLES      = 5000

# ── PCA ───────────────────────────────────────────────────────────────────────
PCA_VARIANCE   = 0.95      # keep components explaining 95% of variance

# ── Training ──────────────────────────────────────────────────────────────────
TEST_SPLIT     = 0.20      # 20% held out for final test
VAL_SPLIT      = 0.15      # 15% of train used for validation during training
EPOCHS         = 200
BATCH_SIZE     = 32
LEARNING_RATE  = 1e-3

# ── Data-requirement experiment fractions ─────────────────────────────────────
DATA_FRACTIONS = [0.05, 0.10, 0.25, 0.50, 0.75, 1.0]

print('✅  Imports done. TensorFlow version:', tf.__version__)

## Cell 3 — Helper Functions

In [ ]:
def metrics_report(y_true, y_pred, label='Model'):
    """Compute and print MAE, RMSE, R² in original (unscaled) units."""
    mae  = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    r2   = r2_score(y_true, y_pred)
    print(f'  📊 {label}')
    print(f'       MAE  : {mae:.6f}')
    print(f'       RMSE : {rmse:.6f}')
    print(f'       R²   : {r2:.6f}')
    return mae, rmse, r2


def build_model(input_dim, hidden=(128, 64, 32), dropout_rate=0.2):
    """
    Regularised MLP for regression.
    • BatchNormalization  → stable, faster convergence
    • Dropout             → prevents overfitting
    • Linear output       → regression (no clipping)
    """
    model = Sequential(name=f'MLP_{input_dim}in')
    model.add(Input(shape=(input_dim,)))
    for units in hidden:
        model.add(Dense(units, activation='relu'))
        model.add(BatchNormalization())
        model.add(Dropout(dropout_rate))
    model.add(Dense(1, activation='linear'))
    model.compile(
        optimizer=Adam(learning_rate=LEARNING_RATE),
        loss='mse',
        metrics=['mae']
    )
    return model


# Shared callbacks for all training runs
def get_callbacks():
    return [
        EarlyStopping(monitor='val_loss', patience=20,
                      restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                          patience=8, min_lr=1e-6, verbose=0)
    ]

print('✅  Helper functions defined.')

## Cell 4 — Load Feature Data (output.csv / output.xlsx)
Each row = one thickness distribution profile (226 nodal values).

In [ ]:
csv_path  = os.path.join(BASE_DIR, 'output.csv')
xlsx_path = os.path.join(ALT_DATA_DIR, 'output.xlsx')

if os.path.exists(csv_path):
    X_raw = pd.read_csv(csv_path, header=None).values.astype(np.float32)
    print('Loaded from output.csv')
elif os.path.exists(xlsx_path):
    X_raw = pd.read_excel(xlsx_path, header=0).values.astype(np.float32)
    print(f'Loaded from {xlsx_path}')
    alt_stress = os.path.join(ALT_DATA_DIR, 'stress')
    if os.path.exists(alt_stress):
        STRESS_DIR = alt_stress
        print(f'Using stress files from {STRESS_DIR}')
else:
    raise FileNotFoundError(
        f'❌ Cannot find output.csv in {BASE_DIR} or output.xlsx in {ALT_DATA_DIR}. '
        'Place the dataset files in one of those folders or update BASE_DIR.'
    )

if not os.path.exists(STRESS_DIR):
    raise FileNotFoundError(
        f'❌ Cannot find stress directory at {STRESS_DIR}. '
        f'Expected it either in {BASE_DIR}/stress or {ALT_DATA_DIR}/stress.'
    )

print(f'Shape : {X_raw.shape}  →  {X_raw.shape[0]} samples  ×  {X_raw.shape[1]} features')

# Quick preview
df_preview = pd.DataFrame(X_raw[:5], columns=[f'node_{i}' for i in range(X_raw.shape[1])])
print('First 5 rows, first 8 columns')
display(df_preview.iloc[:, :8])

## Cell 5 — Extract Max Von Mises Stress (Target Variable)
Reads column 0 (stress values) from each of the 5000 stress text files.

In [ ]:
y_raw     = np.full(N_SAMPLES, np.nan, dtype=np.float32)
bad_files = []

for i in tqdm(range(N_SAMPLES), desc='Reading stress files'):
    fp = os.path.join(STRESS_DIR, f'stress_{i}.txt')
    try:
        data   = np.loadtxt(fp)
        y_raw[i] = data[:, 0].max() if data.ndim > 1 else data.max()
    except Exception as e:
        bad_files.append((i, str(e)))

# Report any bad files
if bad_files:
    print(f'⚠  {len(bad_files)} files could not be read:')
    for idx, err in bad_files[:5]:
        print(f'   stress_{idx}.txt → {err}')

# Drop NaN samples
valid = ~np.isnan(y_raw)
X_raw, y_raw = X_raw[valid], y_raw[valid]

print(f'\nValid samples : {X_raw.shape[0]}')
print(f'Stress range  : [{y_raw.min():.4f},  {y_raw.max():.4f}]')
print(f'Stress mean   : {y_raw.mean():.4f}  ±  {y_raw.std():.4f}')

# Distribution of target
fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(y_raw, bins=60, color='steelblue', edgecolor='white', alpha=0.85)
ax.set_xlabel('Max Von Mises Stress')
ax.set_ylabel('Count')
ax.set_title('Distribution of Max Von Mises Stress (all 5000 samples)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Cell 6 — Train / Test Split & Feature Scaling

In [ ]:
# ── 80/20 split ───────────────────────────────────────────────────────────────
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X_raw, y_raw, test_size=TEST_SPLIT, random_state=SEED
)

# ── Scale features (zero mean, unit variance) ─────────────────────────────────
scaler_X   = StandardScaler()
X_train_sc = scaler_X.fit_transform(X_train_full)
X_test_sc  = scaler_X.transform(X_test)

# ── Scale target (helps gradient descent converge) ────────────────────────────
scaler_y   = StandardScaler()
y_train_sc = scaler_y.fit_transform(y_train_full.reshape(-1, 1)).ravel()
y_test_sc  = scaler_y.transform(y_test.reshape(-1, 1)).ravel()

print(f'Train samples : {X_train_sc.shape[0]}')
print(f'Test  samples : {X_test_sc.shape[0]}')
print(f'Feature dims  : {X_train_sc.shape[1]}')

## Cell 7 — PCA Dimensionality Reduction
Reduces the 226 thickness features down to the smallest number of components that still explain 95% of the variance.

In [ ]:
pca = PCA(n_components=PCA_VARIANCE, random_state=SEED)
X_train_pca = pca.fit_transform(X_train_sc)
X_test_pca  = pca.transform(X_test_sc)

N_COMPONENTS       = X_train_pca.shape[1]
variance_explained = pca.explained_variance_ratio_.sum() * 100

print(f'Original features    : {X_train_sc.shape[1]}')
print(f'PCA components kept  : {N_COMPONENTS}')
print(f'Variance explained   : {variance_explained:.2f}%')
print(f'Compression ratio    : {X_train_sc.shape[1] / N_COMPONENTS:.1f}×')

# ── Plot explained variance ───────────────────────────────────────────────────
cumvar = np.cumsum(pca.explained_variance_ratio_) * 100

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(range(1, len(cumvar)+1), cumvar, color='steelblue', linewidth=1.5)
ax.fill_between(range(1, N_COMPONENTS+1), cumvar[:N_COMPONENTS], alpha=0.15, color='steelblue')
ax.axhline(95, color='red',   linestyle='--', linewidth=1, label='95% threshold')
ax.axvline(N_COMPONENTS, color='green', linestyle=':', linewidth=1.5,
           label=f'{N_COMPONENTS} components selected')
ax.set_xlabel('Number of PCA Components')
ax.set_ylabel('Cumulative Explained Variance (%)')
ax.set_title('PCA: Cumulative Explained Variance')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.bar(range(1, min(31, N_COMPONENTS+1)),
       pca.explained_variance_ratio_[:30] * 100,
       color='steelblue', alpha=0.8)
ax.set_xlabel('PCA Component Index')
ax.set_ylabel('Individual Variance Explained (%)')
ax.set_title('Top 30 PCA Components (individual variance)')
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Cell 8 — Train Approach 1: Direct MLP (226 inputs)

In [ ]:
print('Training Approach 1 — Direct MLP (226 inputs) ...')
t0 = time.time()

model1 = build_model(X_train_sc.shape[1], hidden=(256, 128, 64))
model1.summary()

hist1 = model1.fit(
    X_train_sc, y_train_sc,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VAL_SPLIT,
    callbacks=get_callbacks(),
    verbose=1
)

print(f'\n⏱  Trained {len(hist1.history["loss"])} epochs in {time.time()-t0:.1f}s')

## Cell 9 — Train Approach 2: PCA + MLP

In [ ]:
print(f'Training Approach 2 — PCA ({N_COMPONENTS} inputs) + MLP ...')
t0 = time.time()

model2 = build_model(N_COMPONENTS, hidden=(128, 64, 32))
model2.summary()

hist2 = model2.fit(
    X_train_pca, y_train_sc,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_split=VAL_SPLIT,
    callbacks=get_callbacks(),
    verbose=1
)

print(f'\n⏱  Trained {len(hist2.history["loss"])} epochs in {time.time()-t0:.1f}s')

## Cell 10 — Evaluate Both Models on Test Set

In [ ]:
# Predict and inverse-scale back to original stress units
pred1_raw = scaler_y.inverse_transform(model1.predict(X_test_sc,  verbose=0)).ravel()
pred2_raw = scaler_y.inverse_transform(model2.predict(X_test_pca, verbose=0)).ravel()

print('='*50)
mae1, rmse1, r2_1 = metrics_report(y_test, pred1_raw, 'Approach 1  (Direct MLP, 226 inputs)')
print()
mae2, rmse2, r2_2 = metrics_report(y_test, pred2_raw, f'Approach 2  (PCA {N_COMPONENTS} inputs + MLP)')
print('='*50)

## Cell 11 — Learning Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, hist, title, color in zip(
        axes,
        [hist1, hist2],
        ['Approach 1 · Direct MLP (226 inputs)', f'Approach 2 · PCA ({N_COMPONENTS} inputs) + MLP'],
        ['steelblue', 'darkorange']):
    ax.plot(hist.history['loss'],     label='Train loss', color=color, linewidth=1.5)
    ax.plot(hist.history['val_loss'], label='Val loss',   color='coral', linestyle='--', linewidth=1.5)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('MSE Loss (scaled)')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Learning Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Cell 12 — Predicted vs Actual & Residuals

In [ ]:
mn, mx = y_test.min(), y_test.max()
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Approach 1 — Predicted vs Actual
ax = axes[0]
ax.scatter(y_test, pred1_raw, alpha=0.35, s=8, color='steelblue')
ax.plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Ideal')
ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
ax.set_title(f'Approach 1 · Predicted vs Actual\nR² = {r2_1:.4f}')
ax.legend(); ax.grid(True, alpha=0.3)

# Approach 2 — Predicted vs Actual
ax = axes[1]
ax.scatter(y_test, pred2_raw, alpha=0.35, s=8, color='darkorange')
ax.plot([mn, mx], [mn, mx], 'r--', linewidth=1.5, label='Ideal')
ax.set_xlabel('Actual'); ax.set_ylabel('Predicted')
ax.set_title(f'Approach 2 · Predicted vs Actual\nR² = {r2_2:.4f}')
ax.legend(); ax.grid(True, alpha=0.3)

# Residual histograms overlaid
ax = axes[2]
ax.hist(y_test - pred1_raw, bins=40, alpha=0.6, color='steelblue',
        label='Approach 1', density=True)
ax.hist(y_test - pred2_raw, bins=40, alpha=0.6, color='darkorange',
        label='Approach 2', density=True)
ax.axvline(0, color='red', linestyle='--', linewidth=1)
ax.set_xlabel('Residual  (Actual − Predicted)')
ax.set_ylabel('Density')
ax.set_title('Residual Distribution')
ax.legend(); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Cell 13 — Data-Requirement Experiment
Retrain both models on increasing fractions of training data to see which needs **less data** to perform well.

> ⏳ This cell trains 12 models — may take a few minutes.

In [ ]:
results = {'frac': DATA_FRACTIONS, 'n': [],
           'mae1': [], 'mae2': [],
           'r2_1': [], 'r2_2': []}

for frac in tqdm(DATA_FRACTIONS, desc='Data fraction experiment'):
    n = max(32, int(len(X_train_sc) * frac))
    results['n'].append(n)

    x_tr     = X_train_sc[:n]
    x_tr_pca = X_train_pca[:n]
    y_tr     = y_train_sc[:n]

    # Approach 1
    m1 = build_model(X_train_sc.shape[1], hidden=(256, 128, 64))
    m1.fit(x_tr, y_tr, epochs=EPOCHS, batch_size=min(32, n),
           validation_split=0.1, callbacks=get_callbacks(), verbose=0)
    p1 = scaler_y.inverse_transform(m1.predict(X_test_sc, verbose=0)).ravel()

    # Approach 2
    m2 = build_model(N_COMPONENTS, hidden=(128, 64, 32))
    m2.fit(x_tr_pca, y_tr, epochs=EPOCHS, batch_size=min(32, n),
           validation_split=0.1, callbacks=get_callbacks(), verbose=0)
    p2 = scaler_y.inverse_transform(m2.predict(X_test_pca, verbose=0)).ravel()

    results['mae1'].append(np.mean(np.abs(y_test - p1)))
    results['mae2'].append(np.mean(np.abs(y_test - p2)))
    results['r2_1'].append(r2_score(y_test, p1))
    results['r2_2'].append(r2_score(y_test, p2))

    print(f'  n={n:5d}  |  App1  MAE={results["mae1"][-1]:.4f}  R²={results["r2_1"][-1]:.4f}'
          f'  |  App2  MAE={results["mae2"][-1]:.4f}  R²={results["r2_2"][-1]:.4f}')

## Cell 14 — Data-Requirement Plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(results['n'], results['mae1'], marker='o', linewidth=2,
        color='steelblue', label='Approach 1  (226 dim)')
ax.plot(results['n'], results['mae2'], marker='s', linewidth=2,
        color='darkorange', linestyle='--', label=f'Approach 2  ({N_COMPONENTS} dim PCA)')
ax.set_xlabel('Training Samples Used')
ax.set_ylabel('MAE  (original stress units)')
ax.set_title('Data Requirement  ·  MAE')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(results['n'], results['r2_1'], marker='o', linewidth=2,
        color='steelblue', label='Approach 1  (226 dim)')
ax.plot(results['n'], results['r2_2'], marker='s', linewidth=2,
        color='darkorange', linestyle='--', label=f'Approach 2  ({N_COMPONENTS} dim PCA)')
ax.set_xlabel('Training Samples Used')
ax.set_ylabel('R²')
ax.set_title('Data Requirement  ·  R²')
ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('How much data does each approach need?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Cell 15 — Final Summary Table

In [ ]:
summary = pd.DataFrame({
    'Property': [
        'Input dimensions',
        'Network architecture',
        'MAE  (original units)',
        'RMSE (original units)',
        'R²',
        'Training epochs (final model)'
    ],
    'Approach 1  (Direct MLP)': [
        226,
        '256 → 128 → 64 → 1',
        f'{mae1:.4f}',
        f'{rmse1:.4f}',
        f'{r2_1:.4f}',
        len(hist1.history['loss'])
    ],
    f'Approach 2  (PCA + MLP)': [
        N_COMPONENTS,
        '128 → 64 → 32 → 1',
        f'{mae2:.4f}',
        f'{rmse2:.4f}',
        f'{r2_2:.4f}',
        len(hist2.history['loss'])
    ]
})

display(summary.set_index('Property').style
        .set_caption('Q4 — Final Comparison: Approach 1 vs Approach 2')
        .set_table_styles([{'selector': 'caption',
                            'props': [('font-size', '14px'),
                                      ('font-weight', 'bold')]}]))

print('\n✅  Q4 Complete!')